In [ ]:
# Databricks notebook source
# Kaapi Bricks KA Evaluation
# Experiment: ka-06ac94eb-dev-experiment (ID: 982411422225142)

In [ ]:
%pip install -U 'mlflow[databricks]>=3.1'
%restart_python

In [ ]:
import re
import requests
import mlflow
from mlflow.genai.scorers import Correctness, Safety, RelevanceToQuery, RetrievalGroundedness
from mlflow.entities import Document
from databricks.sdk import WorkspaceClient
from urllib.parse import unquote

KA_ENDPOINT_NAME = "ka-06ac94eb-endpoint"

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(experiment_id="982411422225142")

In [ ]:
@mlflow.trace
def predict(query: str) -> str:
    w = WorkspaceClient()
    url = f"{w.config.host.rstrip('/')}/serving-endpoints/{KA_ENDPOINT_NAME}/invocations"
    headers = w.config.authenticate()
    headers["Content-Type"] = "application/json"
    resp = requests.post(url, headers=headers, json={"input": [{"role": "user", "content": query}]}, timeout=300)
    resp.raise_for_status()
    data = resp.json()

    # KA returns output_text blocks with url_citation annotations.
    # Collect all text parts and deduplicated annotations in one pass.
    text_parts = []
    all_annotations = []
    seen_urls = set()
    for item in data.get("output", []):
        if item.get("type") == "message":
            for block in item.get("content", []):
                if block.get("type") == "output_text":
                    text_parts.append(block.get("text", ""))
                    for ann in block.get("annotations", []):
                        if ann.get("type") == "url_citation" and ann.get("url") not in seen_urls:
                            seen_urls.add(ann["url"])
                            all_annotations.append(ann)

    full_text = "".join(text_parts).strip() or "No response from agent."

    # RetrievalGroundedness requires a function decorated with span_type="RETRIEVER"
    # that returns list[Document]. Define it here so it closes over all_annotations.
    @mlflow.trace(span_type="RETRIEVER", name="retriever")
    def retrieve_ka_docs(q: str) -> list:
        docs = []
        for i, ann in enumerate(all_annotations):
            ann_url = ann.get("url", "")
            source = ann.get("title", ann_url)
            fragment = ann_url.split('#', 1)[1] if '#' in ann_url else ''
            text_match = re.search(r'text=(.+)', fragment)
            content = (
                unquote(text_match.group(1)).replace('%0A', '\n').strip()
                if text_match else source
            )
            if content:
                docs.append(Document(id=str(i), page_content=content, metadata={"source": source}))
        return docs

    if all_annotations:
        retrieve_ka_docs(query)

    return full_text

In [ ]:
eval_dataset = [
    # 1. Classic Filter Coffee — someone asking how to make it
    {
        "inputs": {"query": "How do I make a Classic Filter Coffee?"},
        "expectations": {
            "expected_facts": [
                "Uses 30ml freshly dripped decoction",
                "House blend is 70% Chikmagalur Robusta + 30% Chicory",
                "Brewed at 92-96 degrees C, dripped 12-15 minutes",
                "120ml hot full cream milk",
                "Serve in tumbler-davara with 3-4 pours for froth",
                "Price is Rs.60",
            ],
        },
    },
    # 2. Decoction process — specific how-to question
    {
        "inputs": {"query": "What is the correct decoction ratio and timing?"},
        "expectations": {
            "expected_facts": [
                "2 heaped tablespoons (20g) of house blend per 150ml water",
                "Water temperature 92-96 degrees C",
                "Drip for 12-15 minutes",
                "Maximum hold time is 4 hours",
            ],
        },
    },
    # 3. Allergen handling — safety question
    {
        "inputs": {"query": "What should a barista do if a customer reports a nut allergy?"},
        "expectations": {
            "expected_facts": [
                "Confirm the specific allergen",
                "Check all ingredients including syrups and add-ons",
                "Use freshly cleaned equipment to prevent cross-contact",
                "If unsure, consult the manager before preparing the drink",
            ],
        },
    },
    # 4. Supplier lead time — specific operational question
    {
        "inputs": {"query": "How long does Coorg Arabica take to reorder?"},
        "expectations": {
            "expected_facts": [
                "Lead time is 7 days",
                "Supplier is Coorg Coffee Estates (SUP-001)",
                "Minimum order is 50 kg",
            ],
        },
    },
    # 5. Milk storage — food safety question
    {
        "inputs": {"query": "At what temperature should milk be stored?"},
        "expectations": {
            "expected_facts": [
                "Full Cream Milk must be refrigerated at 2-4 degrees C",
                "Use within 2 days of delivery",
                "Refrigerator must be below 4 degrees C",
            ],
        },
    },
    # 6. Brass filter cleaning — equipment question
    {
        "inputs": {"query": "How do I clean the brass filter coffee maker?"},
        "expectations": {
            "expected_facts": [
                "After each batch: rinse both chambers with hot water",
                "End of day: disassemble, wash with warm soapy water, air dry upside down",
                "Weekly: soak in baking soda solution for 30 minutes",
                "Polish with lemon and salt weekly for brass",
                "Never use abrasive scrubbers",
            ],
        },
    },
    # 7. Franchise — general inquiry
    {
        "inputs": {"query": "How can I open a Kaapi Bricks franchise?"},
        "expectations": {
            "expected_facts": [
                "Kaapi Bricks expands through a franchise model",
                "Available across India and select international markets",
                "Should mention financial requirements or investment needed",
            ],
        },
    },
    # 8. Masala Chai — recipe question
    {
        "inputs": {"query": "How do I make Masala Chai?"},
        "expectations": {
            "expected_facts": [
                "Uses Kerala spice blend with cardamom, ginger, and cinnamon",
                "Price is Rs.50",
                "Spices from Kerala Spice Traders (SUP-005)",
            ],
        },
    },
    # 9. Milk supplier — who and how
    {
        "inputs": {"query": "Who supplies our milk and what are their delivery terms?"},
        "expectations": {
            "expected_facts": [
                "Nandini Dairy (SUP-004)",
                "Supplies Full Cream Milk and Toned Milk",
                "Should mention lead time or delivery schedule",
                "Should mention payment terms",
            ],
        },
    },
    # 10. Shelf life — safety question
    {
        "inputs": {"query": "How long can prepared decoction be kept before discarding?"},
        "expectations": {
            "expected_facts": [
                "Maximum 4 hours for prepared decoction",
                "After 4 hours the coffee oxidizes and tastes stale",
            ],
        },
    },
]

In [ ]:
results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict,
    scorers=[
        Safety(),
        RelevanceToQuery(),
        Correctness(),
        RetrievalGroundedness(),
    ],
)

display(results.tables["eval_results"])